# MORPH — Kaggle training (T4x2, resumable)
Accelerator: GPU T4 x2. Inputs: output of the `morph-data-prep` notebook. Optional secret `HF_TOKEN` (HF write) for checkpoints that survive across sessions.
The CLI path (`scripts/launch_kaggle.py`) does the same thing headlessly.

In [ ]:
!git clone -q -b morph-v2 https://github.com/dhairya-pandya/MoRPH.git /tmp/MoRPH
%cd /tmp/MoRPH
import os, glob
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret:', e)
DATA = os.path.dirname(glob.glob('/kaggle/input/**/meta.json', recursive=True)[0])
print(DATA)

In [ ]:
# one-time: throughput + memory check (~30 min)
!PYTHONPATH=. python -m morph.tools.gpu_gate --out /kaggle/working/gate --quick

In [ ]:
# main run session: checkpoints every 45 min + before the 12 h cap; relaunch to resume
!PYTHONPATH=. torchrun --nproc_per_node=2 -m morph.train.pretrain \
  --model_config configs/model/s.json --train_config configs/train/main_s.json \
  --set data_dir=$DATA --set out_dir=/kaggle/working/runs --set time_limit_hours=11.5